# Evaluación y comparación final

Este notebook reconstruye el flujo de datos y evalúa los cuatro modelos entrenados en el experimento. La comparación se realiza con la misma partición estratificada y sobre la misma base de variables para mantener la comparabilidad.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/features_final.csv")
X = df.drop(columns=["has_chart_hits"])
y = df["has_chart_hits"]
X = X.select_dtypes(include="number")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

relational_features = ["degree", "degree_centrality", "clustering", "pagerank", "community"]
X_train_relacional = X_train[relational_features]
X_test_relacional = X_test[relational_features]

scaler_relacional = StandardScaler()
X_train_rel_scaled = scaler_relacional.fit_transform(X_train_relacional)
X_test_rel_scaled = scaler_relacional.transform(X_test_relacional)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("Relational features:", relational_features)

Train shape: (118778, 7) (118778,)
Test shape: (29695, 7) (29695,)
Relational features: ['degree', 'degree_centrality', 'clustering', 'pagerank', 'community']


## Modelos mixtos y relacionales

En esta sección se evalúan el Random Forest mixto y el Random Forest puramente relacional con la misma división train/test.

In [3]:
results = []

rf_mixed = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_estimators=300,
    n_jobs=-1,
)
rf_mixed.fit(X_train_scaled, y_train)
y_pred_mixed = rf_mixed.predict(X_test_scaled)
cm_mixed = confusion_matrix(y_test, y_pred_mixed)
report_mixed = classification_report(y_test, y_pred_mixed, digits=4, zero_division=0, output_dict=True)
print("Modelo 1 - Random Forest mixto")
print(cm_mixed)
print(classification_report(y_test, y_pred_mixed, digits=4, zero_division=0))
results.append({
    "Modelo": "Modelo 1 - RF mixto",
    "Accuracy": report_mixed["accuracy"],
    "Precision clase 1": report_mixed["1"]["precision"],
    "Recall clase 1": report_mixed["1"]["recall"],
    "F1 clase 1": report_mixed["1"]["f1-score"],
})

rf_rel = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_estimators=100,
    n_jobs=-1,
)
rf_rel.fit(X_train_rel_scaled, y_train)
y_pred_rel = rf_rel.predict(X_test_rel_scaled)
cm_rel = confusion_matrix(y_test, y_pred_rel)
report_rel = classification_report(y_test, y_pred_rel, digits=4, zero_division=0, output_dict=True)
print("\nModelo 2 - Random Forest relacional")
print(cm_rel)
print(classification_report(y_test, y_pred_rel, digits=4, zero_division=0))
results.append({
    "Modelo": "Modelo 2 - RF relacional",
    "Accuracy": report_rel["accuracy"],
    "Precision clase 1": report_rel["1"]["precision"],
    "Recall clase 1": report_rel["1"]["recall"],
    "F1 clase 1": report_rel["1"]["f1-score"],
})

Modelo 1 - Random Forest mixto
[[26385   265]
 [  803  2242]]
              precision    recall  f1-score   support

           0     0.9705    0.9901    0.9802     26650
           1     0.8943    0.7363    0.8076      3045

    accuracy                         0.9640     29695
   macro avg     0.9324    0.8632    0.8939     29695
weighted avg     0.9627    0.9640    0.9625     29695


Modelo 2 - Random Forest relacional
[[26088   562]
 [  814  2231]]
              precision    recall  f1-score   support

           0     0.9697    0.9789    0.9743     26650
           1     0.7988    0.7327    0.7643      3045

    accuracy                         0.9537     29695
   macro avg     0.8843    0.8558    0.8693     29695
weighted avg     0.9522    0.9537    0.9528     29695



## Modelos neuronales

Se evalúan ahora el MLP puramente relacional y el MLP mixto definitivo.

In [4]:
mlp_rel = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=1000,
    early_stopping=True,
    random_state=42,
)
mlp_rel.fit(X_train_rel_scaled, y_train)
y_pred_mlp_rel = mlp_rel.predict(X_test_rel_scaled)
cm_mlp_rel = confusion_matrix(y_test, y_pred_mlp_rel)
report_mlp_rel = classification_report(y_test, y_pred_mlp_rel, digits=4, zero_division=0, output_dict=True)
print("Modelo 3 - MLP relacional")
print(cm_mlp_rel)
print(classification_report(y_test, y_pred_mlp_rel, digits=4, zero_division=0))
results.append({
    "Modelo": "Modelo 3 - MLP relacional",
    "Accuracy": report_mlp_rel["accuracy"],
    "Precision clase 1": report_mlp_rel["1"]["precision"],
    "Recall clase 1": report_mlp_rel["1"]["recall"],
    "F1 clase 1": report_mlp_rel["1"]["f1-score"],
})

mlp_mixed = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=1000,
    early_stopping=True,
    random_state=42,
)
mlp_mixed.fit(X_train_scaled, y_train)
y_pred_mlp_mixed = mlp_mixed.predict(X_test_scaled)
cm_mlp_mixed = confusion_matrix(y_test, y_pred_mlp_mixed)
report_mlp_mixed = classification_report(y_test, y_pred_mlp_mixed, digits=4, zero_division=0, output_dict=True)
print("\nModelo 4 - MLP mixto")
print(cm_mlp_mixed)
print(classification_report(y_test, y_pred_mlp_mixed, digits=4, zero_division=0))
results.append({
    "Modelo": "Modelo 4 - MLP mixto",
    "Accuracy": report_mlp_mixed["accuracy"],
    "Precision clase 1": report_mlp_mixed["1"]["precision"],
    "Recall clase 1": report_mlp_mixed["1"]["recall"],
    "F1 clase 1": report_mlp_mixed["1"]["f1-score"],
})

Modelo 3 - MLP relacional
[[26392   258]
 [  881  2164]]
              precision    recall  f1-score   support

           0     0.9677    0.9903    0.9789     26650
           1     0.8935    0.7107    0.7917      3045

    accuracy                         0.9616     29695
   macro avg     0.9306    0.8505    0.8853     29695
weighted avg     0.9601    0.9616    0.9597     29695


Modelo 4 - MLP mixto
[[26354   296]
 [  792  2253]]
              precision    recall  f1-score   support

           0     0.9708    0.9889    0.9798     26650
           1     0.8839    0.7399    0.8055      3045

    accuracy                         0.9634     29695
   macro avg     0.9274    0.8644    0.8926     29695
weighted avg     0.9619    0.9634    0.9619     29695



## Comparación final

La tabla resume las métricas más relevantes para el problema desbalanceado. En este caso, la clase positiva es la de mayor interés, por lo que se priorizan precision, recall y F1 de la clase 1.

In [5]:
comparison_df = pd.DataFrame(results).set_index("Modelo").sort_values(by="F1 clase 1", ascending=False)
comparison_df.round(4)

,Accuracy,Precision clase 1,Recall clase 1,F1 clase 1
Modelo,,,,
Modelo 1 - RF mixto,0.9640,0.8943,0.7363,0.8076
Modelo 4 - MLP mixto,0.9634,0.8839,0.7399,0.8055
Modelo 3 - MLP relacional,0.9616,0.8935,0.7107,0.7917
Modelo 2 - RF relacional,0.9537,0.7988,0.7327,0.7643
